# Recoverability-guided Go2 transition validation
Python 3.11. Upload recoverability_transition_molab_20261112_v3.zip beside this notebook. Runs bounded MuJoCo validation only; no PPO is launched.

In [ ]:
from pathlib import Path
import os,sys,json,zipfile,hashlib,subprocess
assert sys.version_info[:2] == (3,11)
root=Path("recoverability_transition_workspace")
assert not root.exists(), "Use a new workspace to preserve results"
with zipfile.ZipFile("recoverability_transition_molab_20261112_v3.zip") as z:
    z.extractall(root)
os.chdir(root)
for name,digest in json.loads(Path("transition_source_manifest.json").read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest
subprocess.run([sys.executable,"-m","pip","install","-r","transition_requirements.txt"],check=True)
os.environ["MPLCONFIGDIR"]="/tmp/go2_transition_matplotlib"
os.environ["XDG_CACHE_HOME"]="/tmp/go2_transition_cache"
os.environ["MUJOCO_GL"]="egl"
os.environ["WANDB_MODE"]="disabled"
def run(*args):
    subprocess.run([sys.executable,"-m",*args],check=True)


In [ ]:
run("pytest","-q","experiments/recoverability/test_core.py","experiments/recoverability/test_transition.py")


In [ ]:
run("experiments.recoverability.transition","--bank-only","--output","output/molab_filter54")
print(Path("output/molab_filter54/decision.json").read_text())


In [ ]:
run("experiments.recoverability.validate_transition","--output","output/molab_transition_physics54","--budget","12")
run("experiments.recoverability.report_transition","--output","output/molab_transition_physics54","--physics-analysis","output/molab_transition_physics54/physics_handoff_analysis.json")


In [ ]:
run("experiments.recoverability.ablate_transition","--search","output/molab_transition_physics54","--output","output/molab_transition_ablations54")
run("experiments.recoverability.contact_diagnostics","--output","output/molab_transition_contacts")


No feasible kick-to-balance reference was found in local validation. Two-update transition training and the gated ten-update pilot remain disabled pending a validated transition, held-out recovery tests, and safety-preserving curriculum admission. Reset recovery at 90 degrees is not a standing-to-handstand transition.